# K-Nearest Neighbours (KNN) from Scratch
### Breast cancer classification with NumPy

This notebook implements the **k-nearest neighbours classifier from scratch** (NumPy only) and applies it to the
[Breast Cancer Wisconsin (Diagnostic)](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html) dataset.
scikit-learn is used **only** to load the data and to split it into train and test sets. The scaler, the distance functions,
the neighbour search, the voting and the metrics are all written by hand.

**Result:** 98.2% test accuracy (56 of 57 correct), precision 1.00 and recall 0.95 for malignant tumours.

**Contents**
1. Imports
2. Load the dataset
3. Explore the data
4. Train/test split
5. Feature scaling
6. Distance functions
7. The KNN algorithm
8. Evaluation
9. Results, limitations and next steps

---

## How KNN works

KNN is a **lazy learner**: there is no training step, it just stores the training data. To classify a new point $q$:

1. Compute the distance from $q$ to every training point.
2. Pick the $k$ closest training points (the *neighbours*).
3. Predict the **majority class** among those neighbours.

Because the prediction depends entirely on distances, **feature scaling is essential** (see section 5).

In [ ]:
# Libraries: NumPy does all the maths; pandas is only for inspecting the data;
# scikit-learn is used just to load the dataset and split it.
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

## 2. Load the dataset

The dataset has **569 samples** and **30 numeric features** computed from images of breast mass cell nuclei
(radius, texture, perimeter, area, smoothness, and so on, each as a mean, a standard error, and a "worst" value).

The target is binary:

| Label | Meaning | Count |
|---|---|---|
| `0` | malignant | 212 |
| `1` | benign | 357 |

In [335]:
# Load the Breast Cancer Wisconsin dataset (569 samples, 30 features).
data_ = load_breast_cancer()

In [336]:
# Put the features in a DataFrame so we can inspect them with pandas.
df = pd.DataFrame(data=data_.data, 
                  columns=data_.feature_names)


In [337]:
# Preview the full feature table.
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [338]:
# First five rows.
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Explore the data

Quick checks before modelling: the shape, the data types, missing values and the range of each feature.

In [339]:
# Column types and missing values (there are none: 569 non-null in every column).
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   mean radius              569 non-null    float64
 1   mean texture             569 non-null    float64
 2   mean perimeter           569 non-null    float64
 3   mean area                569 non-null    float64
 4   mean smoothness          569 non-null    float64
 5   mean compactness         569 non-null    float64
 6   mean concavity           569 non-null    float64
 7   mean concave points      569 non-null    float64
 8   mean symmetry            569 non-null    float64
 9   mean fractal dimension   569 non-null    float64
 10  radius error             569 non-null    float64
 11  texture error            569 non-null    float64
 12  perimeter error          569 non-null    float64
 13  area error               569 non-null    float64
 14  smoothness error         569 non-null

In [340]:
# Summary statistics. Note the very different ranges across features,
# which is why scaling is needed later.
df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,16.269190,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,4.833242,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,7.930000,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,13.010000,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,14.970000,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,18.790000,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,36.040000,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500


In [341]:
# (rows, columns) = (569, 30)
df.shape

(569, 30)

In [342]:
# Target labels: 0 = malignant, 1 = benign.
y = data_.target

In [343]:
# Inspect the label array.
y

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0,
       0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0,

## 4. Train/test split

We hold out **10%** of the data (57 samples) as a test set that is used only for the final evaluation.

- `stratify=y` keeps the class ratio (about 37% malignant) the same in both splits, which matters because the classes are imbalanced.
- `random_state=42` makes the split reproducible.

In [344]:
# 90/10 stratified split. Stratifying keeps the malignant/benign ratio
# the same in the train and test sets.
x_train, x_test, y_train, y_test = train_test_split(
    df.values, y,
    test_size=0.1,
    random_state=42,
    stratify= y
)

In [345]:
# Raw (unscaled) training features.
x_train

array([[1.499e+01, 2.211e+01, 9.753e+01, ..., 1.308e-01, 3.163e-01,
        9.251e-02],
       [1.989e+01, 2.026e+01, 1.305e+02, ..., 1.613e-01, 2.549e-01,
        9.136e-02],
       [1.048e+01, 1.986e+01, 6.672e+01, ..., 6.736e-02, 2.883e-01,
        7.748e-02],
       ...,
       [1.114e+01, 1.407e+01, 7.124e+01, ..., 3.922e-02, 2.576e-01,
        7.018e-02],
       [9.876e+00, 1.940e+01, 6.395e+01, ..., 9.749e-02, 2.622e-01,
        8.490e-02],
       [1.065e+01, 2.522e+01, 6.801e+01, ..., 6.136e-02, 3.409e-01,
        8.147e-02]], shape=(512, 30))

In [346]:
# Training labels.
y_train

array([1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1,
       1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1,
       1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1,
       0, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1,
       0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1,

## 5. Feature scaling (standardisation)

KNN uses distances, so features with large numeric ranges dominate. For example, `mean area` is in the hundreds or thousands
while `mean smoothness` is around 0.1. Without scaling, the distance would be almost entirely determined by area-like features.

We **standardise** each feature to zero mean and unit variance:

$$z = \frac{x - \mu}{\sigma}$$

**Important:** $\mu$ and $\sigma$ are computed on the **training set only** and then applied to both sets.
Computing them on the full dataset would leak information from the test set into training.

In [347]:
# Standardise: z = (x - mean) / std.
# Statistics come from the TRAINING set only and are applied to both sets (no data leakage).
# If a feature has zero variance, set its std to 1 to avoid division by zero.
def scale(a_tr, a_te):
    m, s = a_tr.mean(0), a_tr.std(0); s[s == 0] = 1
    return (a_tr - m) / s, (a_te - m) / s

x_train, x_test = scale(x_train, x_test)

In [348]:
# Scaled training features: each column now has mean ~0 and std ~1.
x_train

array([[ 0.25480383,  0.65241465,  0.23804349, ...,  0.25343762,
         0.42062373,  0.47696213],
       [ 1.6538891 ,  0.22812653,  1.60057602, ...,  0.71341983,
        -0.55397506,  0.41334442],
       [-1.03292567,  0.13638856, -1.03522395, ..., -0.70332538,
        -0.02381872, -0.35449371],
       ...,
       [-0.84447745, -1.19151857, -0.84842846, ..., -1.12771553,
        -0.51111811, -0.75832789],
       [-1.20538435,  0.0308899 , -1.14969817, ..., -0.24892328,
        -0.43810257,  0.05597883],
       [-0.98438598,  1.36567738, -0.98191285, ..., -0.79381369,
         0.81109816, -0.13376791]], shape=(512, 30))

In [357]:
# Number of test samples.
y_test.shape

(57,)

In [358]:
# Number of training samples.
y_train.shape

(512,)

## 6. Distance functions

Each function takes the matrix of training points `given_pt` (shape `n × d`) and one query point `query` (shape `d`),
and returns the distance from the query to **every** training point as a vector of length `n`. NumPy broadcasting
removes the need for a Python loop.

| Metric | Formula |
|---|---|
| Euclidean | $d(x,y)=\sqrt{\sum_j (x_j-y_j)^2}$ |
| Manhattan | $d(x,y)=\sum_j \lvert x_j-y_j\rvert$ |
| Minkowski | $d(x,y)=\left(\sum_j \lvert x_j-y_j\rvert^{p}\right)^{1/p}$ |

Minkowski generalises the other two: $p=1$ gives Manhattan and $p=2$ gives Euclidean.

In [349]:
# Euclidean distance from `query` to every row of `given_pt` (straight-line distance).
def euclidian(given_pt, query):
    dist = np.sqrt(((given_pt - query) ** 2).sum(axis = 1))
    return dist

In [350]:
# Manhattan distance (sum of absolute differences).
def manhattan(given_pt, query):
    dist = (abs(given_pt - query)).sum(axis = 1)
    return dist

In [351]:
# Minkowski distance of order p. abs() is needed so odd/fractional powers
# of negative differences do not produce NaN.
# To use it as a dist_fn, fix p first, e.g. functools.partial(minkowski, p=3).
def minkowski(given_pt, query, p):
    dist = ((np.abs(given_pt - query) ** p).sum(axis = 1)) ** (1/p)
    return dist

## 7. The KNN algorithm

Three small functions, each building on the previous one:

- `get_neighbours`: indices of the $k$ training points closest to a query.
- `predict_one`: classifies a single query by **majority vote** among its neighbours.
- `predict`: classifies every row of a test matrix.

`np.bincount(labels).argmax()` counts how many neighbours have each label (index = label) and returns the label with the most votes.
Using an **odd $k$** avoids ties in binary classification.

In [352]:
# Return the indices of the k training points nearest to the query.
# argsort sorts distances ascending, so the first k entries are the nearest.
def get_neighbours(X_train, q, k, dist_fn = euclidian):
    dist = dist_fn(X_train, q)
    return np.argsort(dist)[:k]

In [353]:
# Classify one query point by majority vote among its k nearest neighbours.
# y_train[idx] -> neighbour labels; bincount counts votes per class; argmax picks the winner.
def predict_one(X_train, y_train, q, k, dist_fn=euclidian):
    idx = get_neighbours(X_train, q, k, dist_fn)
    return np.bincount(y_train[idx]).argmax()

In [354]:
# Predict a label for every test point, then measure accuracy.
# k = 5 (odd, so there are no ties in a two-class vote).
def predict(X_train, y_train, X_test, k, dist_fn=euclidian):
    return np.array([predict_one(X_train, y_train, q, k, dist_fn) for q in X_test])

y_pred = predict(x_train, y_train, x_test, k=5)
print("accuracy:", np.mean(y_pred == y_test))

accuracy: 0.9824561403508771


In [355]:
# Predicted labels for the 57 test samples.
y_pred

array([0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1])

## 8. Evaluation

Accuracy alone can hide the type of mistakes. In cancer screening a **missed malignant tumour (false negative)** is far more costly than a false alarm,
so we treat **malignant (`0`) as the positive class** and report the confusion matrix, precision and recall.

|  | Predicted malignant | Predicted benign |
|---|---|---|
| **Actually malignant** | TP | FN |
| **Actually benign** | FP | TN |

$$\text{precision}=\frac{TP}{TP+FP}\qquad \text{recall}=\frac{TP}{TP+FN}$$

- **Precision:** of the tumours predicted malignant, how many really were.
- **Recall:** of the truly malignant tumours, how many were found.

In [356]:
# Confusion-matrix counts with malignant (0) as the positive class.
# Printed as [[TP, FN], [FP, TN]]; rows = actual, columns = predicted.
tp = np.sum((y_pred == 0) & (y_test == 0))
fp = np.sum((y_pred == 0) & (y_test == 1))
fn = np.sum((y_pred == 1) & (y_test == 0))
tn = np.sum((y_pred == 1) & (y_test == 1))
print([[tp, fn], [fp, tn]])
print("precision", tp/(tp+fp), "recall", tp/(tp+fn))

[[np.int64(20), np.int64(1)], [np.int64(0), np.int64(36)]]
precision 1.0 recall 0.9523809523809523


## 9. Results, limitations and next steps

### Results (k = 5, Euclidean distance, standardised features)

| Metric | Value |
|---|---|
| Accuracy | 0.982 (56 / 57) |
| Precision (malignant) | 1.000 |
| Recall (malignant) | 0.952 (20 of 21 malignant tumours found) |

The model made a single error: one malignant tumour was classified as benign.

### Limitations

- **Small test set.** With 57 test samples, one extra mistake changes accuracy by about 1.8 percentage points, so this number is noisy.
- **Single split.** The result comes from one train/test split; a different `random_state` could give a different score.
- **k was not tuned.** $k=5$ is a common default, not a value selected by validation.
- **Prediction cost.** Every prediction computes distances to all training points, which costs $O(n \cdot d)$ per query.

### Possible next steps

- Choose $k$ with **k-fold cross-validation** on the training set and plot accuracy against $k$.
- Compare **Euclidean, Manhattan and Minkowski** distances.
- Add **distance-weighted voting** (closer neighbours count more).
- Show the effect of **not scaling** the features.
- Wrap everything in a `KNN` class with `fit`, `predict` and `predict_proba`.
- Validate against `sklearn.neighbors.KNeighborsClassifier`.